# POD for 0D Isothermal Chemical Reactor
#### Importing libraries

In [ ]:
import os
import re
import sys
import json
import itertools
import numpy as np
import pandas as pd
import joblib as jl
import tensorflow as tf

from tqdm import tqdm
from PCAfold import preprocess, PCA

In [ ]:
sys.path.append('/home/zanardi/Codes/TC/HyperNet/v.22.12/hypernet')
sys.path.append('/home/zanardi/Codes/ML/PyCOMET/pycomet')
from src import utils
from src import error as err_mod
from src import plotting as plt_mod
from src import reduction as red_mod
from datagen.data_generator import DataGenerator

#### Inputs
Parallel operations

In [ ]:
# Max number of cores
nb_jobs = 16

Physics parameters

In [ ]:
# Data ------------------------------------------------------------------------
# Input space (samples)
inp_space = "./datagen/runs/inp_space.csv"

# Time (limit values)
time_limits = [1.e-10, 1.e-6]

# Timescales ------------------------------------------------------------------
groupings = ['RVE1', 'RVE3', 'RVE9', 'RVE27']

groups_graph = {
    0: {
        0: {
            0: [0,1,2],
            1: [3,4,5],
            2: [6,7,8]
        },
        1: {
            3: [9,10,11],
            4: [12,13,14],
            5: [15,16,17],
        },
        2: {
            6: [18,19,20],
            7: [21,22,23],
            8: [24,25,26],
        }
    }
}

# Thermochemistry -------------------------------------------------------------
bins = (1, 3, 9, 27)

specie_thermo = {
    0: {
        'specie': {
            'name': 'O2',
            'model': {
               'levels': 'O3_UMN/O2.csv',
               'lev_mapping': f'O3_UMN/O2/LevelsMap_RVE{bins[-1]}.csv'
            }
        },
        'thermo': {
            'model': 'CoupledModes'
        }
    },
    1: {
        'specie': {
            'name': 'O',
            'model': {
               'levels': 'O3_UMN/O.csv',
            }
        },
        'thermo': {
            'model': 'Constant'
        }
    }
}

mol_specie_thermos = [
    {
        "specie": {
            "name": "O2",
            "model": {
               "levels": "O3_UMN/O2.csv",
               "lev_mapping": f"O3_UMN/O2/LevelsMap_RVE{nb_bins}.csv"
            }
        },
        "thermo": {
            "model": "CoupledModes"
        }
    } for nb_bins in bins
]

process = ['adiabatic', 'isochoric']

# Neq. error ------------------------------------------------------------------
neq_names = ['NEQ1', 'NEQ2', 'NEQ31', 'NEQ32', 'NEQ33']

SVD parameters

In [ ]:
# Data scaling
scaling = 'std'  #['none', 'auto', 'std', 'pareto', 'vast', 'range', '0to1']
# Max variance missing
cum_var_eps = 0.001

In [ ]:
# Define the error minimum value and the list of errors
err_min = 0.02
nb_pc_max = 64

errors = ['mrae', 'mse']
reductions = ['max', 'mean']

Paths

In [ ]:
# Path to outputs directory
path_ot_outputs = './output/'
os.makedirs(path_ot_outputs, exist_ok=True)
# Path to SVD analysis
path_to_svd = path_ot_outputs + f'cumvar.{(1.0-cum_var_eps)*100}'
delta = int(np.absolute(np.diff(np.log10(np.array(time_limits)))))
path_to_svd += f'_t.O({delta})'
os.makedirs(path_to_svd, exist_ok=True)

#### Generate data

Solve ODEs

In [ ]:
dataGen = DataGenerator(
    specie_thermo,
    mol_specie_thermos,
    process,
    groups_graph,
    specie_names={"m": "O2", "a": "O"},
    tlim=time_limits,
    load_plato="module load plato/dev-gcc-9.3.0-openmpi-3.1.3-sundials-5.8.0",
    nb_jobs=nb_jobs
)

ics = pd.read_csv(inp_space)
cases = dataGen.generate(ics)
nb_cases = len(cases)

# Extract data
t = cases[0][0]
y = [case[1] for case in cases]

In [ ]:
# Concatenating
Y = [np.hstack(yi) for yi in list(zip(*y))]
# Preprocessing: removing constant columns
Yr, iremoved, iretained = [tf.nest.map_structure(
    lambda x: red_mod.remove_constant(x)[i], Y
) for i in range(3)]

### SVD analysis

Perform SVD

In [ ]:
print(f'\nPerform SVD')

# Define plotting parameters
eig_min = 1.e-14   # Minimum eigenvalue to be plotted
pc_max = 16        # Maximum component to be plotted

# Define directory
ein_dir = path_to_svd + '/eigen'
os.makedirs(ein_dir, exist_ok=True)

pca = {}
nb_pc_trunc = {}
for i, y_i in enumerate(Yr):
    key = neq_names[i]
    print(f'>> SVD for {key} dynamics')
    print(f'>>    Shapes:', tf.nest.map_structure(np.shape, y_i))
    # Path
    fig_path = ein_dir + '/' + key
    os.makedirs(fig_path, exist_ok=True)
    # Initialize the PCA object
    pca_i = PCA(y_i, scaling=scaling, n_components=nb_pc_max)
    # Singular values
    plt_mod.eigen(
        eigenvalues=pca_i.L,
        xlim=[None, pc_max],
        figname=fig_path+'/eigen.pdf'
    )
    # Cumulative variance
    plt_mod.cumvar(
        eigenvalues=pca_i.L,
        xlim=[None, pc_max],
        eps=cum_var_eps,
        figname=fig_path+'/cumvar.pdf'
    )
    # Collect max number of PC
    cumvar = np.cumsum(pca_i.L, axis=0)
    pc_i = np.where(cumvar > 1.0-cum_var_eps)[0][0]+1
    pca_i.n_components = int(pc_i)
    nb_pc_trunc[key] = int(pc_i)
    # Collect PCA object
    pca[key] = pca_i

with open(ein_dir+"/nb_pc.json", "w") as outfile:
    json.dump(nb_pc_trunc, outfile, indent=4)

Evaluate error

In [ ]:
# Compute
err = jl.Parallel(nb_jobs)(
    jl.delayed(err_mod.compute_err)(
        Y,
        pca,
        nb_pc,
        nb_cases,
        iremoved=iremoved,
        iretained=iretained,
        errors=errors,
        reductions=reductions,
        nb_jobs=nb_jobs
    ) for nb_pc in tqdm(range(1, nb_pc_max+1))
)
# Concatenate
err = utils.concatenate_list_struct(err)

In [ ]:
print(f'\nPlotting error')

# Define directory
err_dir = path_to_svd + '/err'
os.makedirs(err_dir, exist_ok=True)

nb_pc_trunc_max = {}
nb_pc_trunc_mean = {}
for i, y_i in enumerate(Y):
    key = neq_names[i]
    print(f'>> Reconstruction error for {key} dynamics')
    # Define the min number of principal components to get small error
    err_i = {}
    nb_pc_min = {}
    for e in errors:
        err_i[e] = {}
        nb_pc_min[e] = {}
        for r in reductions:
            err_i[e][r] = err[e][r][i]
            nb_pc_min[e][r] = np.argmin(np.abs(err_i[e][r] - err_min)) + 1
    # Plot
    plt_mod.rec_error(
        err_i,
        nb_pc_min,
        nb_pc_max,
        err_min,
        errors=errors,
        reductions=reductions,
        figname=err_dir+f'/{key}.pdf'
    )
    # Number of min pc
    nb_pc_i = {k: int(nb_pc_min['mrae'][k]) for k in ('max', 'mean')}
    nb_pc_trunc_max[key] = nb_pc_i['max']
    nb_pc_trunc_mean[key] = nb_pc_i['mean']
    pca[key].n_components = nb_pc_i['max']

with open(err_dir+"/nb_pc_max.json", "w") as outfile:
    json.dump(nb_pc_trunc_max, outfile, indent=4)

with open(err_dir+"/nb_pc_mean.json", "w") as outfile:
    json.dump(nb_pc_trunc_mean, outfile, indent=4)

Visualize the principal components

In [ ]:
print(f'\nVisualize the principal components')

# Define directory
pc_dir = path_to_svd + '/pc'
os.makedirs(pc_dir, exist_ok=True)

for i, y_i in enumerate(Yr):
    key = neq_names[i]
    print(f'>> PC for {key} dynamics')
    fig_path = pc_dir+'/'+key
    os.makedirs(fig_path, exist_ok=True)
    plt_mod.pca_modes(t, y_i, pca[key], fig_path)

Reconstruct variables

In [ ]:
def plot_rec_scenario(t, y, y_rec, scales, figname):
    for i, y_i in enumerate(y):
        key = neq_names[i]
        # Saving directory
        fig_path = rec_dir+'/'+key
        os.makedirs(fig_path, exist_ok=True)
        # Plotting samples
        plt_mod.rec_solution(
            t=t,
            y=y_i,
            y_rec=y_rec[i],
            figname=fig_path+'/'+figname,
            scales=scales
        )
            
def visualize_scenario(t, y, y_rec, l, nb_cases):
    y_l = utils.extract_case_from_struct(y, l, nb_cases)
    y_rec_l = utils.extract_case_from_struct(y_rec, l, nb_cases)
    # Plot mep anti-xform solution
    plot_rec_scenario(
        t,
        y_l,
        y_rec_l,
        figname='case.'+str(l+1).zfill(3)+'.pdf',
        scales=['log', 'linear']
    )

In [ ]:
print(f'\nReconstruct variables')

nb_samples = 20

# Define directory
rec_dir = path_to_svd + '/rec'
os.makedirs(rec_dir, exist_ok=True)

# Reconstruct
Y_rec = red_mod.reconstruct(
    x=Y,
    pca=pca,
    iremoved=iremoved,
    iretained=iretained
)
idx = np.sort(np.random.choice(
    np.arange(nb_samples), size=nb_samples, replace=False
))

_ = jl.Parallel(nb_jobs)(
    jl.delayed(visualize_scenario)(
        t, Y, Y_rec, l, nb_cases
    ) for l in tqdm(idx)
)